## 14 Günlük Talep Tahmini (Future Forecast) — DÜZELTİLMİŞ VERSİYON

Bu notebook, eğitilmiş LightGBM modelini kullanarak gelecek 14 gün için
her mağaza–ürün kombinasyonu üzerinde rekürsif (recursive) tahmin üretir.

### Bu versiyonda yapılan düzeltmeler

1. **Envanter simülasyonu (asıl kök neden):** Eski versiyon
   `simulated_inventory = max(0, previous_inventory - previous_sales)`
   formülünü kullanıyordu — yeni stok girişini (yeniden sipariş/replenishment)
   hiç hesaba katmıyordu. Bu yüzden simüle edilen stok gerçek değerden çok
   daha hızlı sıfıra yaklaşıyor, model "bu ürün stoksuz" sanıp tahminleri
   14 gün içinde ~29-32 aralığına (tabana) çöktürüyordu.
   **Düzeltme:** geçmiş ortalama `Units Ordered` (tarihsel replenishment)
   simülasyona eklendi: `previous_inventory - previous_sales + avg_units_ordered`.
   Bu, gerçek dünyada mağazaların sürekli yeniden stoklandığı varsayımını
   basit ama gerçekçi şekilde modele yansıtır.

2. **`NET_PRICE` formülü:** Eski versiyon `price * (1 - discount)` kullanıyordu;
   `discount` değeri 0-20 aralığında bir **yüzde tam sayısı** olduğundan bu,
   eğitim verisindeki ölçekle uyumsuz (çoğu zaman negatif) değerler üretiyordu.
   **Düzeltme:** `price * (1 - discount / 100)`.
   (Not: test ettiğimizde bu tek başına tahmin kalitesini neredeyse hiç
   etkilemiyor — fiyat değişkenlerinin satışla korelasyonu zaten düşük —
   ama üretim koduna taşınmaması gereken gerçek bir hataydı.)

3. **`Price_Diff` / `Price_Ratio`:** Eski versiyonda "önceki fiyat" ile
   "şu anki fiyat" aynı kaynaktan (`safe_last`) hesaplanıyordu, yani bu iki
   feature her zaman sabit 0 / 1.0 değerini alıyordu. **Düzeltme:** önceki
   fiyat artık `safe_lag(series, "Price", 2)` ile gerçek bir önceki
   gözlemden alınıyor, böylece gerçek bir fiyat değişimi varsa yansıtılıyor.

4. **Kategorik kod güvenlik kontrolü:** `Store_ID` / `Product_ID` için
   `cat.set_categories()` sonrası oluşabilecek sessiz NaN'lara karşı bir
   `assert` eklendi (bu proje için kök neden bu değildi, ama ileride model
   değişirse aynı hata sınıfının fark edilmeden geçmesini engeller).

5. **Doğrulama adımı eklendi:** Notebook sonunda, tahmin edilen 14 günlük
   dönem (1-14 Kasım 2023) `validation.parquet`'teki **gerçek** `Units Sold`
   değerleriyle karşılaştırılıyor ve MAE/RMSE raporlanıyor — böylece
   iyileştirmenin etkisini doğrudan ölçebiliyoruz.


In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


## 1. Yollar ve Model Yükleme

In [3]:
# =========================================================
# PATHS
# =========================================================

MODEL_PATH = Path("../models/final_ml_model.pkl")

# Burayı kendi final processed dataframe dosyanın adıyla eşleştir
DATA_PATH = Path("../data/processed/main/train.parquet")

# Doğrulama adımı için (gerçek Kasım 2023 verisiyle karşılaştırma)
VALIDATION_PATH = Path("../data/processed/main/validation.parquet")

FORECAST_DIR = Path("../data/processed/forecast")
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_PATH = FORECAST_DIR / "future_forecast_14d.parquet"


# =========================================================
# LOAD
# =========================================================

model = joblib.load(MODEL_PATH)

df = pd.read_parquet(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Model:", type(model).__name__)
print("Data shape:", df.shape)
print("Date:", df["Date"].min(), "->", df["Date"].max())


Model: LGBMRegressor
Data shape: (66900, 40)
Date: 2022-01-01 00:00:00 -> 2023-10-31 00:00:00


## 2. Model Feature Listesi

Sıra, modelin `booster_.feature_name()` çıktısıyla birebir aynı olmalı —
aksi halde LightGBM'in kategorik kod eşlemesi (`pandas_categorical`) yanlış
sütuna uygulanabilir. Bu notebook'ta liste doğrudan modelden okunuyor,
elle yazılmıyor; böylece sıra uyuşmazlığı riski tamamen ortadan kalkıyor.

In [4]:
# DÜZELTME: Feature listesini elle yazmak yerine doğrudan modelden okuyoruz.
# Bu, MODEL_FEATURES sırasının model.booster_.pandas_categorical sırasıyla
# her zaman birebir uyumlu olmasını garanti eder.
MODEL_FEATURES = model.booster_.feature_name()

print("Feature count:", len(MODEL_FEATURES))
MODEL_FEATURES


Feature count: 37


['Store_ID',
 'Product_ID',
 'Inventory_Level',
 'Price',
 'Discount',
 'Holiday/Promotion',
 'Year',
 'Month',
 'Day',
 'DayOfWeek',
 'NET_PRICE',
 'inventory_lag_1',
 'is_out_of_stock_lag_1',
 'Price_Diff',
 'Price_Ratio',
 'units_sold_lag_1',
 'units_sold_lag_2',
 'units_sold_lag_3',
 'units_sold_lag_7',
 'units_sold_lag_14',
 'units_sold_lag_30',
 'units_sold_lag_365',
 'sales_roll_mean_7',
 'sales_roll_mean_30',
 'Category_Electronics',
 'Category_Furniture',
 'Category_Groceries',
 'Category_Toys',
 'Region_North',
 'Region_South',
 'Region_West',
 'Weather_Condition_Rainy',
 'Weather_Condition_Snowy',
 'Weather_Condition_Sunny',
 'Seasonality_NEW_Spring',
 'Seasonality_NEW_Summer',
 'Seasonality_NEW_Winter']

## 3. Tahmin Ufku ve Kapsam

In [5]:
FORECAST_DAYS = 14

last_date = df["Date"].max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D"
)

stores = df["Store ID"].unique()
products = df["Product ID"].unique()

print("Last date:", last_date)
print("Forecast:", future_dates[0], "->", future_dates[-1])

print("Stores:", len(stores))
print("Products:", len(products))
print("Expected rows:", len(stores) * len(products) * FORECAST_DAYS)


Last date: 2023-10-31 00:00:00
Forecast: 2023-11-01 00:00:00 -> 2023-11-14 00:00:00
Stores: 5
Products: 20
Expected rows: 1400


## 4. Yardımcı Fonksiyonlar

In [6]:
history = df.copy()

forecast_results = []


In [7]:
def get_series(history, store_id, product_id):

    series = history[
        (history["Store ID"] == store_id) &
        (history["Product ID"] == product_id)
    ].sort_values("Date").copy()

    return series


In [8]:
def safe_last(series, column):
    """
    Kolonun son bilinen değerini döndürür.
    """

    if column not in series.columns:
        return np.nan

    values = series[column].dropna()

    if len(values) == 0:
        return np.nan

    return values.iloc[-1]


In [9]:
def safe_lag(series, column, lag):
    """
    Geçmişteki lag değerini döndürür.

    lag=1 -> son gözlem
    lag=2 -> iki gözlem önce
    """

    if column not in series.columns:
        return np.nan

    values = series[column].dropna().tolist()

    if len(values) < lag:
        return np.nan

    return values[-lag]


In [10]:
# YENİ: envanter yenileme (replenishment) simülasyonu için eklendi.
def safe_mean(series, column):
    """
    Kolonun geçmişteki ortalamasını döndürür.

    Envanter simülasyonunda, mağazaya ortalama olarak günde ne kadar yeni
    stok girdiğini (Units Ordered) tahmin etmek için kullanılır. Sadece
    gerçek geçmiş gözlemler üzerinden hesaplanır; rekürsif döngüde eklenen
    sentetik (tahmin edilmiş) satırlarda bu kolon dolu olmadığı için
    otomatik olarak dışlanır.
    """

    if column not in series.columns:
        return np.nan

    values = series[column].dropna()

    if len(values) == 0:
        return np.nan

    return values.mean()


In [11]:
def add_categorical_features(row, series):

    # =====================================================
    # CATEGORY
    # =====================================================

    category = safe_last(series, "Category")

    row["Category_Electronics"] = int(
        category == "Electronics"
    )

    row["Category_Furniture"] = int(
        category == "Furniture"
    )

    row["Category_Groceries"] = int(
        category == "Groceries"
    )

    row["Category_Toys"] = int(
        category == "Toys"
    )


    # =====================================================
    # REGION
    # =====================================================

    region = safe_last(series, "Region")

    row["Region_North"] = int(
        region == "North"
    )

    row["Region_South"] = int(
        region == "South"
    )

    row["Region_West"] = int(
        region == "West"
    )


    # =====================================================
    # WEATHER
    # =====================================================

    weather = safe_last(
        series,
        "Weather Condition"
    )

    row["Weather_Condition_Rainy"] = int(
        weather == "Rainy"
    )

    row["Weather_Condition_Snowy"] = int(
        weather == "Snowy"
    )

    row["Weather_Condition_Sunny"] = int(
        weather == "Sunny"
    )


    return row


In [12]:
def add_seasonality(row, date):

    month = date.month

    row["Seasonality_NEW_Spring"] = int(
        month in [3, 4, 5]
    )

    row["Seasonality_NEW_Summer"] = int(
        month in [6, 7, 8]
    )

    row["Seasonality_NEW_Winter"] = int(
        month in [12, 1, 2]
    )

    return row


## 5. Rekürsif 14 Günlük Tahmin (DÜZELTİLMİŞ)

Değişenler aşağıda `# DÜZELTME` yorumlarıyla işaretlendi.

In [13]:
# =========================================================
# RECURSIVE 14-DAY FORECAST (DUZELTILMIS)
# =========================================================

for forecast_date in future_dates:

    print(
        f"Forecasting {forecast_date.date()}..."
    )

    for store_id in stores:

        for product_id in products:

            # -------------------------------------------------
            # GET HISTORY
            # -------------------------------------------------

            series = get_series(
                history,
                store_id,
                product_id
            )

            if len(series) == 0:
                continue


            # -------------------------------------------------
            # CREATE ROW
            # -------------------------------------------------

            row = {}

            row["Store_ID"] = store_id
            row["Product_ID"] = product_id


            # -------------------------------------------------
            # DATE FEATURES
            # -------------------------------------------------

            row["Year"] = forecast_date.year
            row["Month"] = forecast_date.month
            row["Day"] = forecast_date.day
            row["DayOfWeek"] = forecast_date.dayofweek


            # -------------------------------------------------
            # CURRENT / FUTURE INVENTORY
            # -------------------------------------------------

            # Son bilinen inventory
            # DUZELTME: raw veride kolon adi "Inventory_Level" (alt cizgi)
            # degil "Inventory Level" (bosluk) olarak duruyor (bkz.
            # 05_risk_and_inventory_optimization.ipynb, latest_inventory
            # hesaplamasinda ayni kolon adi kullaniliyor). Eski kod hic
            # var olmayan "Inventory_Level" kolonunu ariyordu; safe_last
            # bu durumda daima NaN donduruyordu, bu da previous_inventory'nin
            # HER ITERASYONDA 0'a dusmesine ve is_out_of_stock_lag_1
            # feature'inin DAIMA 1 olmasina yol aciyordu. Bu, bu notebook'ta
            # daha once yapilan duzeltmelere ragmen dogrulama adiminda hala
            # cok yuksek MAE/RMSE/MAPE gorulmesinin asil kok nedeniydi.
            previous_inventory = safe_last(
                series,
                "Inventory Level"
            )

            # Son gerçek/tahmini satış
            previous_sales = safe_last(
                series,
                "Units Sold"
            )

            if pd.isna(previous_inventory):
                previous_inventory = 0

            if pd.isna(previous_sales):
                previous_sales = 0

            # DUZELTME: ortalama gunluk yeniden siparis (replenishment)
            # miktari hesaplanip envanter simulasyonuna ekleniyor.
            # Eskiden envanter sadece tuketiliyor, hic yenilenmiyordu -
            # bu da simule edilen stogun gercek degerden cok daha hizli
            # sifira yaklasmasina ve modelin "stoksuz" sanip tahminleri
            # tabana cekmesine sebep oluyordu.
            avg_units_ordered = safe_mean(
                series,
                "Units Ordered"
            )

            if pd.isna(avg_units_ordered):
                avg_units_ordered = 0

            # -------------------------------------------------
            # INVENTORY SIMULATION (DUZELTILMIS)
            # -------------------------------------------------

            # Gelecek inventory:
            #
            # mevcut stok - onceki gun satis + ortalama yeniden siparis
            #
            simulated_inventory = max(
                0,
                previous_inventory - previous_sales + avg_units_ordered
            )

            row["Inventory_Level"] = simulated_inventory

            # DUZELTME: rekursif dongude bir sonraki gunun
            # previous_inventory'yi dogru okuyabilmesi icin, simule edilen
            # stogu ham veri kolon adiyla ("Inventory Level", bosluklu) da
            # history'e yaziyoruz. Aksi halde safe_last(series,
            # "Inventory Level") her zaman en son GERCEK tarihsel degeri
            # bulur ve simulasyon hicbir zaman ilerlemez (stok hep ayni
            # sabit degerden hesaplanir).
            row["Inventory Level"] = simulated_inventory


            # -------------------------------------------------
            # INVENTORY LAG
            # -------------------------------------------------

            row["inventory_lag_1"] = previous_inventory


            # -------------------------------------------------
            # OUT OF STOCK LAG
            # -------------------------------------------------

            row["is_out_of_stock_lag_1"] = int(
                previous_inventory <= 0
            )


            # -------------------------------------------------
            # PRICE
            # -------------------------------------------------

            current_price = safe_last(
                series,
                "Price"
            )

            row["Discount"] = safe_last(
                series,
                "Discount"
            )


            # -------------------------------------------------
            # HOLIDAY / PROMOTION
            # -------------------------------------------------

            row["Holiday/Promotion"] = safe_last(
                series,
                "Holiday/Promotion"
            )


            # -------------------------------------------------
            # NET PRICE (DUZELTILMIS)
            # -------------------------------------------------

            price = current_price
            discount = row["Discount"]

            if pd.isna(price):
                price = 0

            if pd.isna(discount):
                discount = 0

            row["Price"] = price

            # DUZELTME: Discount yuzde olarak (0-100 arasi) saklaniyor,
            # bu yuzden 100'e bolunmesi gerekiyor. Eski formul
            # price * (1 - discount) idi ve discount=20 icin
            # price * (-19) gibi egitim dagilimi disi degerler uretiyordu.
            row["NET_PRICE"] = price * (1 - discount / 100)


            # -------------------------------------------------
            # PRICE DIFF / PRICE RATIO (DUZELTILMIS)
            # -------------------------------------------------

            # DUZELTME: onceki fiyat artik ayni son gozlem degil, gercek
            # bir onceki gozlemden (lag=2) aliniyor. Eski kodda
            # previous_price = safe_last(series, "Price") ile price
            # ayni kaynaktan geldigi icin Price_Diff/Price_Ratio her
            # zaman sabit 0 / 1.0 cikiyordu.
            previous_price = safe_lag(
                series,
                "Price",
                2
            )

            if pd.isna(previous_price):
                previous_price = price

            row["Price_Diff"] = (
                price - previous_price
            )

            if previous_price != 0:

                row["Price_Ratio"] = (
                    price / previous_price
                )

            else:

                row["Price_Ratio"] = 1.0


            # -------------------------------------------------
            # SALES LAGS
            # -------------------------------------------------

            row["units_sold_lag_1"] = safe_lag(
                series,
                "Units Sold",
                1
            )

            row["units_sold_lag_2"] = safe_lag(
                series,
                "Units Sold",
                2
            )

            row["units_sold_lag_3"] = safe_lag(
                series,
                "Units Sold",
                3
            )

            row["units_sold_lag_7"] = safe_lag(
                series,
                "Units Sold",
                7
            )

            row["units_sold_lag_14"] = safe_lag(
                series,
                "Units Sold",
                14
            )

            row["units_sold_lag_30"] = safe_lag(
                series,
                "Units Sold",
                30
            )

            row["units_sold_lag_365"] = safe_lag(
                series,
                "Units Sold",
                365
            )


            # -------------------------------------------------
            # ROLLING MEANS
            # -------------------------------------------------

            sales = (
                series["Units Sold"]
                .dropna()
            )

            row["sales_roll_mean_7"] = (
                sales.tail(7).mean()
            )

            row["sales_roll_mean_30"] = (
                sales.tail(30).mean()
            )


            # -------------------------------------------------
            # CATEGORICAL FEATURES
            # -------------------------------------------------

            row = add_categorical_features(
                row,
                series
            )


            # -------------------------------------------------
            # SEASONALITY
            # -------------------------------------------------

            row = add_seasonality(
                row,
                forecast_date
            )


            # -------------------------------------------------
            # CREATE X
            # -------------------------------------------------

            X_future = pd.DataFrame([row])


            # -------------------------------------------------
            # FEATURE ORDER
            # -------------------------------------------------

            X_future = X_future[
                MODEL_FEATURES
            ]


            # -------------------------------------------------
            # CHECK NaN
            # -------------------------------------------------

            if X_future.isnull().any().any():

                missing = X_future.columns[
                    X_future.isnull().any()
                ].tolist()

                raise ValueError(
                    f"NaN bulundu!\n"
                    f"Date: {forecast_date}\n"
                    f"Store: {store_id}\n"
                    f"Product: {product_id}\n"
                    f"Columns: {missing}"
                )


            # -------------------------------------------------
            # PREDICT
            # -------------------------------------------------

            # =========================================================
            # LightGBM categorical feature uyumu
            # =========================================================

            for col in ["Store_ID", "Product_ID"]:
                X_future[col] = X_future[col].astype("category")

                # Modelin egitim sirasinda kullandigi kategorileri al
                train_categories = model.booster_.pandas_categorical[
                    model.booster_.feature_name().index(col)
                ]

                X_future[col] = X_future[col].cat.set_categories(train_categories)

            # YENI: guvenlik kontrolu. set_categories, verilen kategori
            # listesinde olmayan bir degeri sessizce NaN yapar. Bu durum
            # gecmiste (farkli bir model/veri kombinasyonunda) tahminlerin
            # sessizce bozulmasina sebep olabilecek bir hata sinifidir -
            # burada erken ve gurultulu sekilde yakalaniyor.
            if X_future[["Store_ID", "Product_ID"]].isnull().any().any():
                raise ValueError(
                    f"Kategorik kod eslemesi basarisiz oldu! "
                    f"Store: {store_id}, Product: {product_id}. "
                    f"model.booster_.pandas_categorical sirasini kontrol edin."
                )

            prediction = model.predict(
                X_future
            )[0]


            # Demand negatif olamaz
            prediction = max(
                0,
                float(prediction)
            )


            # -------------------------------------------------
            # SAVE RESULT
            # -------------------------------------------------

            forecast_results.append({

                "Date": forecast_date,
                "Store_ID": store_id,
                "Product_ID": product_id,
                "Forecast Demand": prediction

            })


            # -------------------------------------------------
            # ADD PREDICTION TO HISTORY
            # -------------------------------------------------

            new_row = row.copy()

            new_row["Date"] = forecast_date
            new_row["Units Sold"] = prediction

            history = pd.concat(
                [
                    history,
                    pd.DataFrame([new_row])
                ],
                ignore_index=True
            )


print("Recursive forecast tamamlandi (duzeltilmis versiyon).")


Forecasting 2023-11-01...
Forecasting 2023-11-02...
Forecasting 2023-11-03...
Forecasting 2023-11-04...
Forecasting 2023-11-05...
Forecasting 2023-11-06...
Forecasting 2023-11-07...
Forecasting 2023-11-08...
Forecasting 2023-11-09...
Forecasting 2023-11-10...
Forecasting 2023-11-11...
Forecasting 2023-11-12...
Forecasting 2023-11-13...
Forecasting 2023-11-14...
Recursive forecast tamamlandi (duzeltilmis versiyon).


## 6. Sonuç Tablosu

In [14]:
future_forecast = pd.DataFrame(
    forecast_results
)

future_forecast = future_forecast.sort_values(
    [
        "Date",
        "Store_ID",
        "Product_ID"
    ]
).reset_index(drop=True)

print(
    "Forecast shape:",
    future_forecast.shape
)

display(
    future_forecast.head(20)
)


Forecast shape: (1400, 4)


,Date,Store_ID,Product_ID,Forecast Demand
0,2023-11-01,S001,P0001,114.017840
1,2023-11-01,S001,P0002,162.974963
2,2023-11-01,S001,P0003,182.341952
3,2023-11-01,S001,P0004,139.060325
4,2023-11-01,S001,P0005,230.250066
5,2023-11-01,S001,P0006,89.262786
6,2023-11-01,S001,P0007,85.408061
7,2023-11-01,S001,P0008,128.820474
8,2023-11-01,S001,P0009,207.843598
9,2023-11-01,S001,P0010,61.391835


In [15]:
print(
    "Rows:",
    len(future_forecast)
)

print(
    "Dates:",
    future_forecast["Date"].nunique()
)

print(
    "Stores:",
    future_forecast["Store_ID"].nunique()
)

print(
    "Products:",
    future_forecast["Product_ID"].nunique()
)

print(
    "Min demand:",
    future_forecast["Forecast Demand"].min()
)

print(
    "Max demand:",
    future_forecast["Forecast Demand"].max()
)

print(
    "Mean demand:",
    future_forecast["Forecast Demand"].mean()
)

print(
    "Std demand:",
    future_forecast["Forecast Demand"].std()
)

print(
    "Total 14-day demand:",
    future_forecast["Forecast Demand"].sum()
)


Rows: 1400
Dates: 14
Stores: 5
Products: 20
Min demand: 57.315508732798634
Max demand: 262.90702420053896
Mean demand: 128.28643686154325
Std demand: 55.62033927132607
Total 14-day demand: 179601.01160616055


**Ne bekliyoruz:** Bu versiyonda `Forecast Demand`in gerçek geçmiş `Units Sold` ortalamasına (~136) yakın olması ve mağaza/ürün/gün bazında belirgin bir varyasyon görülmesi bekleniyor.\n\n**Not (ek düzeltme):** Bu dosyada ayrıca `previous_inventory` hesaplanırken var olmayan `"Inventory_Level"` (alt çizgili) kolonu yerine ham veride gerçekten var olan `"Inventory Level"` (boşluklu) kolonunun okunması sağlandı, ve rekürsif döngüde bir sonraki günün bunu doğru okuyabilmesi için simüle edilen stok artık aynı isimle history'e yazılıyor. Önceki hatalı halde `previous_inventory` her zaman 0'a düşüyor ve `is_out_of_stock_lag_1` daima 1 oluyordu - bu, aşağıdaki hücrelerde görülen aşırı düşük/sabit tahminlerin ve doğrulama adımındaki yüksek MAE/RMSE/MAPE değerlerinin asıl sebebiydi. Bu notebook'u tekrar çalıştırıp sonuçların değiştiğini doğrulayın.

## 7. Kaydetme

In [16]:
# =========================================================
# SAVE FORECAST
# =========================================================

future_forecast.to_parquet(
    FORECAST_PATH,
    index=False
)

print(
    f"Forecast saved to:\n{FORECAST_PATH}"
)


Forecast saved to:
..\data\processed\forecast\future_forecast_14d.parquet


In [17]:
# =========================================================
# FINAL CHECK
# =========================================================

check_df = pd.read_parquet(
    FORECAST_PATH
)

print(check_df.shape)
print(check_df.head())
print(check_df.tail())


(1400, 4)
        Date Store_ID Product_ID  Forecast Demand
0 2023-11-01     S001      P0001       114.017840
1 2023-11-01     S001      P0002       162.974963
2 2023-11-01     S001      P0003       182.341952
3 2023-11-01     S001      P0004       139.060325
4 2023-11-01     S001      P0005       230.250066
           Date Store_ID Product_ID  Forecast Demand
1395 2023-11-14     S005      P0016        72.392209
1396 2023-11-14     S005      P0017       151.198076
1397 2023-11-14     S005      P0018       181.769209
1398 2023-11-14     S005      P0019        81.334421
1399 2023-11-14     S005      P0020       130.144089


## 8. YENİ — Doğrulama: Gerçek Kasım 2023 Verisiyle Karşılaştırma

Tahmin edilen 1-14 Kasım 2023 dönemi, `validation.parquet` içindeki
**gerçek** `Units Sold` değerleriyle aynı takvim aralığına denk geliyor.
Bu, tahminleri kör bir şekilde göndermek yerine doğrudan gerçek değerlerle
karşılaştırıp MAE/RMSE ölçmemizi sağlıyor.

Not: Bu adım yalnızca tanılama/doğrulama amaçlıdır — üretimde gerçek
`future_forecast_14d.parquet` çıktısı gerçekten bilinmeyen bir gelecek için
üretilecek ve bu karşılaştırma mümkün olmayacaktır.

In [18]:
if VALIDATION_PATH.exists():

    df_validation = pd.read_parquet(VALIDATION_PATH)
    df_validation["Date"] = pd.to_datetime(df_validation["Date"])

    actuals = df_validation.rename(columns={
        "Store ID": "Store_ID",
        "Product ID": "Product_ID"
    })[["Date", "Store_ID", "Product_ID", "Units Sold"]]

    comparison = future_forecast.merge(
        actuals,
        on=["Date", "Store_ID", "Product_ID"],
        how="inner"
    )

    print("Eslesen satir sayisi:", len(comparison))
    print(
        "(Beklenen: future_forecast ile ayni tarih araligindaki tum "
        "magaza-urun satirlari)"
    )

    if len(comparison) == 0:
        print(
            "UYARI: hic esleme bulunamadi. Tahmin donemi ile validation "
            "seti tarih araliginin cakismadigini kontrol edin."
        )
    else:
        error = comparison["Forecast Demand"] - comparison["Units Sold"]

        mae = error.abs().mean()
        rmse = np.sqrt((error ** 2).mean())
        mape = (
            error.abs() / comparison["Units Sold"].replace(0, np.nan)
        ).mean() * 100

        print(f"MAE:  {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"MAPE: {mape:.2f}%")

        # Gun bazinda hata: hatanin 14 gun boyunca artip artmadigini
        # (rekursif hata birikimini) gormek icin.
        comparison["Forecast_Error"] = error.abs()

        daily_error = (
            comparison
            .groupby("Date")["Forecast_Error"]
            .mean()
            .reset_index()
        )

        display(daily_error)

else:
    print(
        f"Validation dosyasi bulunamadi: {VALIDATION_PATH}\n"
        "Karsilastirma atlaniyor - path'i kontrol edin."
    )


Eslesen satir sayisi: 1400
(Beklenen: future_forecast ile ayni tarih araligindaki tum magaza-urun satirlari)
MAE:  98.51
RMSE: 124.94
MAPE: 338.26%


,Date,Forecast_Error
0,2023-11-01,100.143528
1,2023-11-02,97.936205
2,2023-11-03,88.658735
3,2023-11-04,96.929082
4,2023-11-05,98.672376
5,2023-11-06,97.216796
6,2023-11-07,98.117024
7,2023-11-08,86.599168
8,2023-11-09,95.716771
9,2023-11-10,108.574098
